In [32]:
import sys
import os
from boto3.session import Session


# Get the current notebook's directory
current_dir = os.path.dirname(os.path.abspath('__file__' if '__file__' in globals() else '.'))

utils_dir = os.path.join(current_dir, '..')
utils_dir = os.path.abspath(utils_dir)

# Add to sys.path
sys.path.insert(0, utils_dir)
print("sys.path[0]:", sys.path[0])
from utils import setup_cognito_user_pool, reauthenticate_user

boto_session = Session()
region = boto_session.region_name
print(f"Region: {region}")

sys.path[0]: D:\Code\github\Bedrock_agent
Region: ap-southeast-1


In [33]:
print("Setting up Amazon Cognito user pool...")
cognito_config = setup_cognito_user_pool("Cognito_3LO_Github")
print("Cognito setup completed ✓")

Setting up Amazon Cognito user pool...
Pool id: ap-southeast-1_Ko1s4ooNU
Discovery URL: https://cognito-idp.ap-southeast-1.amazonaws.com/ap-southeast-1_Ko1s4ooNU/.well-known/openid-configuration
Client ID: 5oef1r9utdhdhen13aoc7cnf1r
Bearer Token: eyJraWQiOiJ4bGFDTUZqcFprWWtFbnpHelE1TVcrM3VrUlFvV05hZnVLS09OT0lPVm5nPSIsImFsZyI6IlJTMjU2In0.eyJzdWIiOiIzOTVhNTU1Yy03MDAxLTcwNTEtZTBkNC0yOTMwMWI4YzdkOGEiLCJpc3MiOiJodHRwczpcL1wvY29nbml0by1pZHAuYXAtc291dGhlYXN0LTEuYW1hem9uYXdzLmNvbVwvYXAtc291dGhlYXN0LTFfS28xczRvb05VIiwiY2xpZW50X2lkIjoiNW9lZjFyOXV0ZGhkaGVuMTNhb2M3Y25mMXIiLCJvcmlnaW5fanRpIjoiMjg0ZTQ3YmEtNmEwZS00MzA3LWE4NmEtNGE5MmZmMGMzYmRjIiwiZXZlbnRfaWQiOiJjNTU4Y2JlZi1jZGQwLTQ4OGYtYWUwNS1hYTQyMzAwNDhjYTYiLCJ0b2tlbl91c2UiOiJhY2Nlc3MiLCJzY29wZSI6ImF3cy5jb2duaXRvLnNpZ25pbi51c2VyLmFkbWluIiwiYXV0aF90aW1lIjoxNzYzNDU5MzU5LCJleHAiOjE3NjM0NjI5NTksImlhdCI6MTc2MzQ1OTM1OSwianRpIjoiMDRiNGEyMzctNzMyNS00NjJhLWExZjItYTA1ZmI0ZDlkMDUyIiwidXNlcm5hbWUiOiJ0ZXN0dXNlciJ9.us2ge2fwlM7LMjFOhzFo_GdT-DqmUy8yK5pGU8A3-QoIlchM

In [34]:
provider_name = "github-provider"

In [36]:
import os
from bedrock_agentcore.services.identity import IdentityClient
from dotenv import load_dotenv
region = boto_session.region_name
identity_client = IdentityClient(region)

load_dotenv()

github_client_id = os.environ["GITHUB_CLIENT_ID"]
github_client_secret = os.environ["GITHUB_CLIENT_SECRET"]

# Configure GitHub OAuth2 provider - On-Behalf-Of User

github_provider = identity_client.create_oauth2_credential_provider({
    "name": provider_name,
    "credentialProviderVendor": "GithubOauth2",
    "oauth2ProviderConfigInput": {
        "githubOauth2ProviderConfig": {
            'clientId': github_client_id,
            'clientSecret': github_client_secret
        }
    }
})
print(github_provider)
print("\n")
print(f"callbackUrl: {github_provider['callbackUrl']}")

{'ResponseMetadata': {'RequestId': '951c562a-c7a2-43f5-b4fd-e25ceb6ca543', 'HTTPStatusCode': 201, 'HTTPHeaders': {'date': 'Tue, 18 Nov 2025 09:50:21 GMT', 'content-type': 'application/json', 'content-length': '934', 'connection': 'keep-alive', 'x-amzn-requestid': '951c562a-c7a2-43f5-b4fd-e25ceb6ca543', 'x-amzn-remapped-x-amzn-requestid': 'f54b21bd-4ae8-4007-929c-4b14623f9ac7', 'x-amzn-remapped-content-length': '934', 'x-amzn-remapped-connection': 'keep-alive', 'x-amz-apigw-id': 'UO8moHHIyQ0EeWA=', 'x-amzn-trace-id': 'Root=1-691c415d-72344a5745027a0965fd7860', 'x-amzn-remapped-date': 'Tue, 18 Nov 2025 09:50:21 GMT'}, 'RetryAttempts': 0}, 'clientSecretArn': {'secretArn': 'arn:aws:secretsmanager:ap-southeast-1:566801649110:secret:bedrock-agentcore-identity!default/oauth2/github-provider-WNiaGp'}, 'name': 'github-provider', 'credentialProviderArn': 'arn:aws:bedrock-agentcore:ap-southeast-1:566801649110:token-vault/default/oauth2credentialprovider/github-provider', 'callbackUrl': 'https://b

In [37]:
# Get the OAuth2 callback URL based on the current environment (notebook/SageMaker)
# This is evaluated HERE in the notebook, not in the agent container
from oauth2_callback_server import get_oauth2_callback_url
oauth2_callback_url_for_agent = get_oauth2_callback_url()

print(f"Callback URL for agent (determined from notebook environment): {oauth2_callback_url_for_agent}")

# Write github_agent.py with the callback URL embedded as a string literal
github_agent_code = f'''
import asyncio
import json
import os
from typing import Optional

import httpx
from bedrock_agentcore import BedrockAgentCoreApp
from bedrock_agentcore.identity.auth import requires_access_token
from strands import Agent, tool

# Environment configuration
os.environ["STRANDS_OTEL_ENABLE_CONSOLE_EXPORT"] = "true"
os.environ["OTEL_PYTHON_EXCLUDED_URLS"] = "/ping,/invocations"

# Global token storage (could be improved with a proper state management solution)
github_access_token: Optional[str] = None

app = BedrockAgentCoreApp()


@tool
def inspect_github_repos() -> str:
    """Inspect and list the user's private GitHub repositories.

    Returns:
        str: A JSON string containing the list of repositories and their details,
            or an authentication required message.
    """
    global github_access_token

    if not github_access_token:
        return json.dumps({{
            "auth_required": True,
            "message": "GitHub authentication is required. Please wait while we set up the authorization.",
            "events": []
        }})

    print(f"Using GitHub access token: {{github_access_token[:10]}}...")

    headers = {{"Authorization": f"Bearer {{github_access_token}}"}}

    try:
        with httpx.Client() as client:
            # Get user information
            user_response = client.get("https://api.github.com/user", headers=headers)
            user_response.raise_for_status()
            username = user_response.json().get("login", "Unknown")
            print(f"✅ User: {{username}}")

            # Search for user's repositories
            repos_response = client.get(
                f"https://api.github.com/search/repositories?q=user:{{username}}",
                headers=headers
            )
            repos_response.raise_for_status()
            repos_data = repos_response.json()
            print(f"✅ Found {{len(repos_data.get('items', []))}} repositories")

            repos = repos_data.get('items', [])
            if not repos:
                return f"No repositories found for {{username}}."

            # Format repository information
            response_lines = [f"GitHub repositories for {{username}}:\\n"]

            for repo in repos:
                repo_line = f"📁 {{repo['name']}}"
                if repo.get('language'):
                    repo_line += f" ({{repo['language']}})"
                repo_line += f" - ⭐ {{repo['stargazers_count']}}"
                response_lines.append(repo_line)

                if repo.get('description'):
                    response_lines.append(f"   {{repo['description']}}")
                response_lines.append("")  # Empty line for spacing

            return "\\n".join(response_lines)

    except httpx.HTTPStatusError as e:
        return f"GitHub API error: {{e.response.status_code}} - {{e.response.text}}"
    except Exception as e:
        return f"Error fetching GitHub repositories: {{str(e)}}"


class StreamingQueue:
    """Simple async queue for streaming responses."""

    def __init__(self):
        self._queue = asyncio.Queue()
        self._finished = False

    async def put(self, item: str) -> None:
        """Add an item to the queue."""
        await self._queue.put(item)

    async def finish(self) -> None:
        """Mark the queue as finished and add sentinel value."""
        self._finished = True
        await self._queue.put(None)

    async def stream(self):
        """Stream items from the queue until finished."""
        while True:
            item = await self._queue.get()
            if item is None and self._finished:
                break
            yield item


# Initialize streaming queue
queue = StreamingQueue()


async def on_auth_url(url: str) -> None:
    """Callback for authentication URL."""
    print(f"Authorization URL: {{url}}")
    await queue.put(f"Authorization URL: {{url}}")


def extract_response_text(response) -> str:
    """Extract text content from agent response."""
    if isinstance(response.message, dict):
        content = response.message.get('content', [])
        if isinstance(content, list):
            return "".join(
                item.get('text', '') for item in content
                if isinstance(item, dict) and 'text' in item
            )
    return str(response.message)


def needs_authentication(response_text: str) -> bool:
    """Check if response indicates authentication is required."""
    auth_keywords = [
        "authentication", "authorize", "authorization", "auth",
        "sign in", "login", "access", "permission", "credential",
        "need authentication", "requires authentication"
    ]
    return any(keyword.lower() in response_text.lower() for keyword in auth_keywords)


async def agent_task(user_message: str) -> None:
    """Execute agent task with authentication handling."""
    global github_access_token

    try:
        await queue.put("Begin agent execution")

        # Initial agent call
        response = agent(user_message)
        response_text = extract_response_text(response)

        # Check if authentication is needed
        if needs_authentication(response_text):
            await queue.put("Authentication required for GitHub access. Starting authorization flow...")

            try:
                github_access_token = await need_token_3LO_async(access_token='')
                await queue.put("Authentication successful! Retrying GitHub request...")

                # Retry with authentication
                response = agent(user_message)
            except Exception as auth_error:
                print(f"Authentication error: {{auth_error}}")
                await queue.put(f"Authentication failed: {{str(auth_error)}}")
                return

        await queue.put(response.message)
        await queue.put("End agent execution")

    except Exception as e:
        await queue.put(f"Error: {{str(e)}}")
    finally:
        await queue.finish()


@requires_access_token(
    provider_name="{provider_name}",
    scopes=["repo", "read:user"],
    auth_flow='USER_FEDERATION',
    on_auth_url=on_auth_url,
    force_authentication=False,  # ← Changed to False - will use cached token!
    callback_url="{oauth2_callback_url_for_agent}"  # ← Callback URL determined from notebook environment
)
async def need_token_3LO_async(*, access_token: str) -> str:
    """Handle 3LO authentication flow."""
    global github_access_token
    github_access_token = access_token
    return access_token


# Create agent instance
agent = Agent(
    model="us.anthropic.claude-3-7-sonnet-20250219-v1:0",
    tools=[inspect_github_repos],
    system_prompt="""You are a GitHub assistant. Use the inspect_github_repos tool to fetch private repositories data.
    The inspect_github_repos tool handles token exchange and proper authentication with the GitHub API
    to obtain private information for the user."""
)


@app.entrypoint
async def agent_invocation(payload):
    """Main entrypoint for agent invocation."""
    user_message = payload.get(
        "prompt",
        "No prompt found in input, please guide customer to create a JSON payload with prompt key"
    )

    # Create and start the agent task
    task = asyncio.create_task(agent_task(user_message))

    async def stream_with_task():
        """Stream results while ensuring task completion."""
        async for item in queue.stream():
            yield item
        await task  # Ensure task completes

    return stream_with_task()


if __name__ == "__main__":
    app.run()
'''

# Write the file
with open("github_agent.py", "w", encoding="utf-8") as f:
    f.write(github_agent_code)

print("✅ github_agent.py written successfully with embedded callback URL")

Callback URL for agent (determined from notebook environment): http://localhost:9090/oauth2/callback
✅ github_agent.py written successfully with embedded callback URL


In [38]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name
print(f"Region: {region}")

discovery_url = cognito_config.get("discovery_url")
client_id = cognito_config.get("client_id")

agentcore_runtime = Runtime()

response = agentcore_runtime.configure(
    entrypoint="github_agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name="strands_agent_github",
    authorizer_configuration={
        "customJWTAuthorizer": {
            "discoveryUrl": discovery_url,
            "allowedClients": [client_id]
        }
    }
)
response

Entrypoint parsed: file=D:\Code\github\Bedrock_agent\03-AgentCore-identity\06-Outbound_Auth_Github\github_agent.py, bedrock_agentcore_name=github_agent
Configuring BedrockAgentCore agent: strands_agent_github


Region: ap-southeast-1


⚠️  ℹ️  No container engine found (Docker/Finch/Podman not installed)
✅ Default deployment uses CodeBuild (no container engine needed)
💡 Run 'agentcore launch' for cloud-based building and deployment
💡 For local builds, install Docker, Finch, or Podman

⚠️  [WARNING] Platform mismatch: Current system is 'linux/amd64' but Bedrock AgentCore requires 'linux/arm64'.
For deployment options and workarounds, see: 
https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/getting-started-custom.html

Generated Dockerfile: D:\Code\github\Bedrock_agent\03-AgentCore-identity\06-Outbound_Auth_Github\Dockerfile
Generated .dockerignore: D:\Code\github\Bedrock_agent\03-AgentCore-identity\06-Outbound_Auth_Github\.dockerignore
Keeping 'strands_agent_github' as default agent
Bedrock AgentCore configured: D:\Code\github\Bedrock_agent\03-AgentCore-identity\06-Outbound_Auth_Github\.bedrock_agentcore.yaml


ConfigureResult(config_path=WindowsPath('D:/Code/github/Bedrock_agent/03-AgentCore-identity/06-Outbound_Auth_Github/.bedrock_agentcore.yaml'), dockerfile_path=WindowsPath('D:/Code/github/Bedrock_agent/03-AgentCore-identity/06-Outbound_Auth_Github/Dockerfile'), dockerignore_path=WindowsPath('D:/Code/github/Bedrock_agent/03-AgentCore-identity/06-Outbound_Auth_Github/.dockerignore'), runtime='None', region='ap-southeast-1', account_id='566801649110', execution_role=None, ecr_repository=None, auto_create_ecr=True)

In [39]:
!cat .bedrock_agentcore.yaml

'cat' �����ڲ����ⲿ���Ҳ���ǿ����еĳ���
���������ļ���


In [40]:
from oauth2_callback_server import get_oauth2_callback_url

# Deploy the agent to AgentCore Runtime and get deployment details
launch_result = agentcore_runtime.launch()
print(launch_result)

if launch_result.agent_id:
    # Extract the workload name from the deployed agent's ID for identity management
    workload_name = launch_result.agent_id
    # Retrieve the current workload identity configuration from AgentCore Identity
    workload_identity = identity_client.get_workload_identity(name=workload_name)
    # Extract existing OAuth2 callback URLs that are already registered for this workload
    allowed_resource_oauth_2_return_urls = workload_identity.get("allowedResourceOauth2ReturnUrls") or []
    # Get the local OAuth2 callback server URL for session binding (localhost:9090/oauth2/callback)
    oauth2_callback_url = get_oauth2_callback_url()
    print(f"Updating workload {workload_name} with callback url {oauth2_callback_url}")

    # Register the local callback URL with the workload identity to enable OAuth2 session binding
    updated_workload_identity = identity_client.update_workload_identity(
        name=workload_name,
        allowed_resource_oauth_2_return_urls=[*allowed_resource_oauth_2_return_urls, oauth2_callback_url],
    )
    print(updated_workload_identity)

🚀 CodeBuild mode: building in cloud (RECOMMENDED - DEFAULT)
   • Build ARM64 containers in the cloud with CodeBuild
   • No local Docker required
💡 Available deployment modes:
   • runtime.launch()                           → CodeBuild (current)
   • runtime.launch(local=True)                 → Local development
   • runtime.launch(local_build=True)           → Local build + cloud deploy (NEW)
Starting CodeBuild ARM64 deployment for agent 'strands_agent_github' to account 566801649110 (ap-southeast-1)
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: strands_agent_github
✅ ECR repository available: 566801649110.dkr.ecr.ap-southeast-1.amazonaws.com/bedrock-agentcore-strands_agent_github
Getting or creating execution role for agent: strands_agent_github
Using AWS region: ap-southeast-1, account ID: 566801649110
Role name: AmazonBedrockAgentCoreSDKRuntime-ap-southeast-1-fdd34a21fd


✅ Reusing existing ECR repository: 566801649110.dkr.ecr.ap-southeast-1.amazonaws.com/bedrock-agentcore-strands_agent_github


✅ Reusing existing execution role: arn:aws:iam::566801649110:role/AmazonBedrockAgentCoreSDKRuntime-ap-southeast-1-fdd34a21fd
✅ Execution role available: arn:aws:iam::566801649110:role/AmazonBedrockAgentCoreSDKRuntime-ap-southeast-1-fdd34a21fd
Preparing CodeBuild project and uploading source...
Getting or creating CodeBuild execution role for agent: strands_agent_github
Role name: AmazonBedrockAgentCoreSDKCodeBuild-ap-southeast-1-fdd34a21fd
Reusing existing CodeBuild execution role: arn:aws:iam::566801649110:role/AmazonBedrockAgentCoreSDKCodeBuild-ap-southeast-1-fdd34a21fd
Using .dockerignore with 44 patterns
Uploaded source to S3: strands_agent_github/source.zip
Updated CodeBuild project: bedrock-agentcore-strands_agent_github-builder
Starting CodeBuild build (this may take several minutes)...
Starting CodeBuild monitoring...
🔄 QUEUED started (total: 0s)
✅ QUEUED completed in 1.2s
🔄 PROVISIONING started (total: 1s)
✅ PROVISIONING completed in 7.3s
🔄 DOWNLOAD_SOURCE started (total: 9s)


mode='codebuild' tag='bedrock_agentcore-strands_agent_github:latest' env_vars=None port=None runtime=None ecr_uri='566801649110.dkr.ecr.ap-southeast-1.amazonaws.com/bedrock-agentcore-strands_agent_github' agent_id='strands_agent_github-UswKnG2MNk' agent_arn='arn:aws:bedrock-agentcore:ap-southeast-1:566801649110:runtime/strands_agent_github-UswKnG2MNk' codebuild_id='bedrock-agentcore-strands_agent_github-builder:fac77390-4a83-44dd-a2c2-0f2930872c11' build_output=None
Updating workload strands_agent_github-UswKnG2MNk with callback url http://localhost:9090/oauth2/callback
{'ResponseMetadata': {'RequestId': '2e5eeb97-cf21-4342-b520-abacb78e9ffd', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Tue, 18 Nov 2025 09:53:44 GMT', 'content-type': 'application/json', 'content-length': '348', 'connection': 'keep-alive', 'x-amzn-requestid': '2e5eeb97-cf21-4342-b520-abacb78e9ffd'}, 'RetryAttempts': 0}, 'name': 'strands_agent_github-UswKnG2MNk', 'workloadIdentityArn': 'arn:aws:bedrock-agentcore:ap-s

In [41]:
import json
import boto3
agentcore_control_client = boto3.client(
    'bedrock-agentcore-control',
    region_name=region
)

runtime_response = agentcore_control_client.get_agent_runtime(
    agentRuntimeId=launch_result.agent_id
)
runtime_role = runtime_response['roleArn']

policies_to_add = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "BedrockModelAccess",
            "Effect": "Allow",
            "Action": [
                "aws-marketplace:ViewSubscriptions",
                "aws-marketplace:Subscribe"
            ],
            "Resource": "*"
        }
    ]
}
iam_client = boto3.client(
    'iam',
    region_name=region
)

response = iam_client.put_role_policy(
    PolicyDocument=json.dumps(policies_to_add),
    PolicyName="outbound_policies",
    RoleName=runtime_role.split("/")[1],
)

In [42]:
import time

status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(status)
print(f"Final status: {status}")

Retrieved Bedrock AgentCore status for: strands_agent_github


Final status: READY


In [43]:
import subprocess
from oauth2_callback_server import store_token_in_oauth2_callback_server, wait_for_oauth2_server_to_be_ready

bearer_token = reauthenticate_user(cognito_config.get("client_id"))

oauth2_callback_server_cmd = [sys.executable, "oauth2_callback_server.py", "--region", region]
oauth2_callback_server_process = subprocess.Popen(oauth2_callback_server_cmd)

try:
    # Start the OAuth2 callback server
    successfully_started_oauth2_server = wait_for_oauth2_server_to_be_ready()
    if not successfully_started_oauth2_server:
        print("Failed to start OAuth2 callback server to handle session binding "
              "(https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/oauth2-authorization-url-session-binding.html)")
    else:
        store_token_in_oauth2_callback_server(bearer_token)
        invoke_response = agentcore_runtime.invoke(
            {"prompt": "What are my private repositories?"},
            bearer_token=bearer_token
        )
        print(invoke_response)
finally:
    oauth2_callback_server_process.terminate()

Timeout: OAuth2 callback server not ready after 40 seconds


Failed to start OAuth2 callback server to handle session binding (https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/oauth2-authorization-url-session-binding.html)
